In [2]:
import os, sys, time, json, warnings, traceback, importlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score, balanced_accuracy_score
HERE = Path(globals().get("__vsc_ipynb_file__", globals().get("__file__", Path.cwd()))).resolve()
try: ROOT = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], cwd=HERE, capture_output=True, text=True, check=True).stdout.strip())
except Exception: ROOT = next(p for p in [HERE, *HERE.parents] if (p / ".git").exists() or ((p / "src").is_dir() and (p / "research").is_dir()))
PROJECT_ROOT, DATA_ROOT = ROOT, ROOT / "data"
sys.path[:0] = [str(ROOT), str(ROOT / "data")]
JFP = next(ROOT.rglob("jerichon_feature_pipeline.py"))
jfp = importlib.import_module(".".join(JFP.relative_to(ROOT).with_suffix("").parts))
HISTORY_ROOT = min({p.parent for p in DATA_ROOT.rglob("realtime") if p.is_dir()}, key=lambda d: "sample_data" in d.parts)
print(PROJECT_ROOT, HISTORY_ROOT, jfp.__file__, sep="\n")

C:\JERICHON-Git\JERICHON
C:\JERICHON-Git\JERICHON\data
C:\JERICHON-Git\JERICHON\src\data\builders\jerichon_feature_pipeline.py


# JERICHON — Trade Flow → Price Response (GAZP)
Исследование отдельных исполнений `trades`, без Dollar Bars, Super Candles и ML.

**Признаки:** прошлые окна 10 / 25 / 50 / 100 сделок (окно 1000 удалено как слишком длинное для текущего этапа). **Target:** цена через 1 / 2 / 3 / 5 / 10 следующих сделок; первое следующее изменение transaction price; цена после 2 / 3 / 5 различных изменений transaction price.

**Важные ограничения:** transaction price ≠ mid-price. Без синхронного best bid/ask нельзя утверждать, что прогнозируется mid-price. `BUYSELL=B/S` нельзя считать стороной агрессора, пока это не подтверждено документацией источника. TFI (дисбаланс исполненных сделок) ≠ OFI (поток изменений заявок/котировок).

### Перекрытие событий
Будущие интервалы target перекрываются. На этом этапе мы не обучаем модель и не применяем uniqueness weights к основным описательным средним: это изменило бы вес наблюдений и сам оцениваемый эффект. Но считаем AFML-style average uniqueness для диагностики и сохраняем веса. Неопределённость оцениваем bootstrap-ом целыми торговыми днями, не iid bootstrap-ом отдельных сделок. Для будущего supervised ML нужны uniqueness weights вместе с purging/embargo и временным split.

## Признаки:
#### Базовые данные сделки
- price = цена исполнения сделки.
- size = объём сделки в единицах, используемых в выгрузке.
- notional = денежный оборот сделки, стоимость исполнения.
- sign = направление сделки в закодированном виде: +1 или −1. Пока не подтверждено, что это именно сторона агрессора.
- signed_size = знаковый объём: направление × объём. В текущем экспорте обнаружена ошибка масштаба.
- signed_notional = знаковый денежный оборот: направление × денежный оборот.
#### Доходность и время
- trade_return_bps = доходность между текущей и предыдущей сделкой в базисных пунктах.
- seconds_since_prev_trade = количество секунд с предыдущей сделки.
- trade_intensity_10_per_sec = интенсивность сделок в окне 10, сделок в секунду.
- trade_intensity_25_per_sec = интенсивность сделок в окне 25, сделок в секунду.
- trade_intensity_50_per_sec = интенсивность сделок в окне 50, сделок в секунду.
- trade_intensity_100_per_sec = интенсивность сделок в окне 100, сделок в секунду.
- trade_intensity_1000_per_sec = интенсивность сделок в окне 1000, сделок в секунду.
- median_gap_10_sec = медианный интервал между сделками в окне 10, секунд.
- median_gap_25_sec = медианный интервал между сделками в окне 25, секунд.
- median_gap_50_sec = медианный интервал между сделками в окне 50, секунд.
- median_gap_100_sec = медианный интервал между сделками в окне 100, секунд.
- median_gap_1000_sec = медианный интервал между сделками в окне 1000, секунд.
#### Trade Flow Imbalance — дисбаланс потока исполнений
- TFI = Trade Flow Imbalance, дисбаланс исполненных сделок. Это не классический OFI, поскольку мы пока не учитываем добавление и отмену заявок.
- tfi_volume_10 = дисбаланс объёма в последних 10 сделках.
- tfi_volume_25 = дисбаланс объёма в последних 25 сделках.
- tfi_volume_50 = дисбаланс объёма в последних 50 сделках.
- tfi_volume_100 = дисбаланс объёма в последних 100 сделках.
- tfi_volume_1000 = дисбаланс объёма в последних 1000 сделках.
- tfi_notional_10 = дисбаланс денежного оборота в последних 10 сделках.
- tfi_notional_25 = дисбаланс денежного оборота в последних 25 сделках.
- tfi_notional_50 = дисбаланс денежного оборота в последних 50 сделках.
- tfi_notional_100 = дисбаланс денежного оборота в последних 100 сделках.
- tfi_notional_1000 = дисбаланс денежного оборота в последних 1000 сделках.
- tfi_count_10 = дисбаланс количества сделок в последних 10 сделках.
- tfi_count_25 = дисбаланс количества сделок в последних 25 сделках.
- tfi_count_50 = дисбаланс количества сделок в последних 50 сделках.
- tfi_count_100 = дисбаланс количества сделок в последних 100 сделках.
- tfi_count_1000 = дисбаланс количества сделок в последних 1000 сделках.

**Для всех этих признаков положительное значение означает преобладание направления, закодированного как +1, а отрицательное — −1. Называть это покупательским или продавцовским давлением можно только после подтверждения семантики BUYSELL.**

#### Направленные серии
- directional_run_10 = длина текущей непрерывной серии сделок одного направления, ограниченная 10.
- directional_run_25 = то же, ограничение 25.
- directional_run_50 = то же, ограничение 50.
- directional_run_100 = то же, ограничение 100.
- directional_run_1000 = то же, ограничение 1000.

**Знак показывает направление серии, модуль — её длину.**

#### Целевые переменные — будущая реакция цены
- ret = return, доходность.
- bps = basis points, базисные пункты. Один базисный пункт равен 0,01%, то есть 1/10 000.
- ret_next_1_trades_bps = изменение transaction price через одну следующую сделку.
- ret_next_3_trades_bps = изменение через три следующие сделки.
- ret_next_5_trades_bps = изменение через пять следующих сделок.
- ret_next_10_trades_bps = изменение через десять следующих сделок.
- ret_next_25_trades_bps = изменение через 25 следующих сделок.
- ret_next_50_trades_bps = изменение через 50 следующих сделок.
- ret_after_1_distinct_price_changes_bps = доходность после первого следующего изменения transaction price.
- ret_after_2_distinct_price_changes_bps = доходность после второго различного изменения цены.
- ret_after_3_distinct_price_changes_bps = доходность после третьего различного изменения цены.
- ret_after_5_distinct_price_changes_bps = доходность после пятого различного изменения цены.
- trades_until_next_price_change = количество сделок до первого изменения цены.
- seconds_until_next_price_change = время до первого изменения цены, секунд.
- next_price_change_direction = направление первого изменения цены: +1, −1 или 0.

#### Контроль прошлой динамики
- past_return_5_trades_bps = доходность за предыдущие 5 сделок.
- past_return_20_trades_bps = доходность за предыдущие 20 сделок.

#### Уникальность событий
- uniqueness = средняя уникальность наблюдения с учётом перекрытия будущих интервалов.
- uniqueness_fixed_1 = уникальность события с горизонтом 1 следующая сделка.
- uniqueness_fixed_3 = уникальность горизонта 3 сделки.
- uniqueness_fixed_5 = уникальность горизонта 5 сделок.
- uniqueness_fixed_10 = уникальность горизонта 10 сделок.
- uniqueness_fixed_25 = уникальность горизонта 25 сделок.
- uniqueness_fixed_50 = уникальность горизонта 50 сделок.
- uniqueness_distinct_1 = уникальность события до первого изменения цены.
- uniqueness_distinct_2 = уникальность до второго изменения цены.
- uniqueness_distinct_3 = уникальность до третьего изменения цены.
- uniqueness_distinct_5 = уникальность до пятого изменения цены.

**Важное уточнение: ret_next_* и ret_after_* — это доходности transaction price, а не mid-price. Это принципиально разные величины.**

## 1. Загрузка trades
Поддерживаются Parquet и CSV. Автопоиск — только удобство; если каталог AlgoPack устроен иначе, укажи `TRADES_FILE` явно.

In [4]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

# ---------- CONFIG ----------
TICKER = "GAZP"

TRADES_FILE = None           # если хочешь один конкретный файл — укажи здесь
BUYSELL_IS_AGGRESSOR = False
SOURCE_TIMEZONE = None
FEATURE_WINDOWS = (10, 25, 50, 100)
TRADE_HORIZONS = (1, 3, 5, 10, 25, 50)
PRICE_CHANGE_HORIZONS = (1, 2, 3, 5)
BOOTSTRAP_REPS = 500
RANDOM_SEED = 63
OUTPUT_DIR = PROJECT_ROOT / "research" / "tradeflow" / "mark_2" / "results" / f"trade_flow_outputs_{TICKER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def discover_trade_file(root: Path, ticker: str) -> Path | None:
    """Найти ОДИН trades-файл (например, для smoke-теста)."""
    if not root.exists():
        return None
    found = []
    for pat in ("**/*trades*.parquet", "**/*trade*.parquet",
                "**/*trades*.csv", "**/*trade*.csv"):
        found.extend(root.glob(pat))
    found = [p for p in found if ticker.upper() in str(p).upper()]
    found.sort(key=lambda p: (ticker.upper() not in p.name.upper(), len(str(p))))
    return found[0] if found else None


def load_all_trades(history_root: Path, ticker: str) -> pd.DataFrame:
    """Собрать все trades.parquet по всем датам для тикера."""
    ticker_dir = history_root / "realtime" / ticker.upper()
    if not ticker_dir.exists():
        raise FileNotFoundError(
            f"Папка тикера не найдена: {ticker_dir}\n"
            f"Проверь, что HISTORY_ROOT указывает на корень данных (data/ или .../storage/history)"
        )

    frames = []
    dates = sorted(
        d.name for d in ticker_dir.iterdir()
        if d.is_dir() and len(d.name) == 10 and d.name[4] == "-" and d.name[7] == "-"
    )
    if not dates:
        raise FileNotFoundError(f"Нет сохранённых дат для {ticker} в {ticker_dir}")

    for day in dates:
        p = ticker_dir / day / "trades.parquet"
        if p.exists():
            try:
                df = pd.read_parquet(p)
                if not df.empty:
                    df = df.copy()
                    df["file_date"] = day
                    frames.append(df)
            except Exception as e:
                print(f"WARNING: не удалось прочитать {p}: {e}")

    if not frames:
        raise FileNotFoundError(f"Не найден ни один trades.parquet для {ticker}")

    out = pd.concat(frames, ignore_index=True)
    print(f"Собрано {len(dates)} дат: {dates[0]} → {dates[-1]}")
    print(f"Всего строк trades: {len(out):,}")
    return out


def load_frame(path):
    path = Path(path)
    if path.suffix.lower() in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file: {path}")

# ---------- LOAD ----------
if TRADES_FILE is not None:
    trade_path = Path(TRADES_FILE)
    if not trade_path.exists():
        raise FileNotFoundError(f"Указанный TRADES_FILE не существует: {trade_path}")
    raw = load_frame(trade_path)
    print("File:", trade_path.resolve(), "shape:", raw.shape)
else:
    raw = load_all_trades(HISTORY_ROOT, TICKER)

display(raw.head(10))
print("Columns:", raw.columns.tolist())

Собрано 33 дат: 2026-09-24 → 2026-11-19
Всего строк trades: 3,097,105


,TRADENO,TRADETIME,PRICE,QUANTITY,PERIOD,VALUE,BUYSELL,DECIMALS,TRADINGSESSION,TRADEDATE,TRADE_SESSION_DATE,ts,ticker,file_date
0,17771970079,06:59:40,102.7,12,S,12324.0,B,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
1,17771970080,06:59:40,102.7,50,S,51350.0,S,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
2,17771970081,06:59:40,102.7,3,S,3081.0,S,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
3,17771970082,06:59:40,102.7,1,S,1027.0,B,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
4,17771970083,06:59:40,102.7,10,S,10270.0,B,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
5,17771970084,06:59:40,102.7,72,S,73944.0,B,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
6,17771970085,06:59:40,102.7,4,S,4108.0,B,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
7,17771970086,06:59:40,102.7,2,S,2054.0,S,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
8,17771970087,06:59:40,102.7,1,S,1027.0,S,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24
9,17771970088,06:59:40,102.7,1,S,1027.0,S,2,0,2026-09-24,2026-09-24,2026-09-24 06:59:40+03:00,GAZP,2026-09-24


Columns: ['TRADENO', 'TRADETIME', 'PRICE', 'QUANTITY', 'PERIOD', 'VALUE', 'BUYSELL', 'DECIMALS', 'TRADINGSESSION', 'TRADEDATE', 'TRADE_SESSION_DATE', 'ts', 'ticker', 'file_date']


## 2. Нормализация и контроль качества
Ожидаемые поля по ранее присланному примеру: `TRADENO`, `ts`/`TRADETIME`, `PRICE`, `QUANTITY`, `VALUE`, `BUYSELL`, `TRADEDATE`/`TRADE_SESSION_DATE`.

In [5]:
def pick_col(columns, candidates, required=True):
    m = {str(c).lower(): c for c in columns}
    for x in candidates:
        if x.lower() in m: return m[x.lower()]
    if required: raise KeyError(f"Нет ни одного поля {candidates}; columns={list(columns)}")
    return None

c_id = pick_col(raw.columns, ["TRADENO", "trade_no", "trade_id"], False)
c_ts = pick_col(raw.columns, ["ts", "TRADETIME", "tradetime", "timestamp"])
c_price = pick_col(raw.columns, ["price", "PRICE"])
c_size = pick_col(raw.columns, ["size", "QUANTITY", "quantity", "volume"])
c_value = pick_col(raw.columns, ["VALUE", "value", "notional"], False)
c_side = pick_col(raw.columns, ["sign", "BUYSELL", "buysell", "side"], False)
c_date = pick_col(raw.columns, ["TRADE_SESSION_DATE", "TRADEDATE", "trade_date", "date"], False)
c_ticker = pick_col(raw.columns, ["ticker", "SECID"], False)

tr = pd.DataFrame(index=raw.index)
tr["trade_id"] = raw[c_id] if c_id else np.arange(len(raw), dtype=np.int64)
tr["ts"] = pd.to_datetime(raw[c_ts], errors="coerce")
if SOURCE_TIMEZONE is not None and tr["ts"].dt.tz is None:
    tr["ts"] = tr["ts"].dt.tz_localize(SOURCE_TIMEZONE)
tr["price"] = pd.to_numeric(raw[c_price], errors="coerce")
tr["size"] = pd.to_numeric(raw[c_size], errors="coerce")
tr["notional"] = (pd.to_numeric(raw[c_value], errors="coerce").abs() if c_value else tr["price"] * tr["size"])
if c_side:
    side = raw[c_side]
    side_text = side.astype("string").str.strip().str.upper()
    if side_text.isin(["B", "S"]).any():
        # AlgoPack BUYSELL: B=+1, S=-1. Other values become NaN, never arbitrary multipliers.
        tr["sign"] = side_text.map({"B": 1.0, "S": -1.0})
    else:
        numeric_side = pd.to_numeric(side, errors="coerce")
        # Accept numeric side only when it is explicitly encoded as -1/+1.
        tr["sign"] = numeric_side.where(numeric_side.isin([-1, 1]), np.nan)
else:
    tr["sign"] = np.nan
tr["trade_date"] = (pd.to_datetime(raw[c_date], errors="coerce").dt.date.astype("string") if c_date else tr["ts"].dt.date.astype("string"))
tr["ticker"] = raw[c_ticker].astype(str) if c_ticker else TICKER
tr = tr.replace([np.inf, -np.inf], np.nan).dropna(subset=["ts", "price", "size"])
tr = tr[(tr.price > 0) & (tr.size > 0)].sort_values(["ts", "trade_id"], kind="mergesort")
dup_ids = int(tr.trade_id.duplicated().sum()) if c_id else 0
if c_id: tr = tr.drop_duplicates("trade_id", keep="last")
tr = tr.reset_index(drop=True)
tr["notional"] = tr.notional.fillna(tr.price * tr.size)
# A signed trade size must be exactly +/- size (or NaN if side is unknown).
tr["sign"] = tr["sign"].where(tr["sign"].isin([-1.0, 1.0]), np.nan)
tr["signed_size"] = tr["size"] * tr["sign"]
tr["signed_notional"] = tr["notional"] * tr["sign"]

# Hard data-integrity guard: prevents a broken sign/size column from silently
# producing impossible TFI values outside [-1, +1].
valid_side = tr["sign"].notna()
if valid_side.any():
    mismatch = ~np.isclose(
        tr.loc[valid_side, "signed_size"].to_numpy(dtype=float),
        (tr.loc[valid_side, "size"] * tr.loc[valid_side, "sign"]).to_numpy(dtype=float),
        rtol=1e-10, atol=1e-10
    )
    if mismatch.any():
        raise ValueError("signed_size != sign * size; остановка до расчёта TFI.")


print(f"rows={len(tr):,}; duplicate IDs removed={dup_ids:,}; days={tr.trade_date.nunique()}; unknown side={tr.sign.isna().mean():.2%}")
display(pd.Series({"unique_trade_id": tr.trade_id.nunique(), "unique_ts": tr.ts.nunique(), "duplicate_ts_rows": tr.ts.duplicated(keep=False).sum(), "ts_monotonic": tr.ts.is_monotonic_increasing, "unique_prices": tr.price.nunique()}).to_frame("value"))
display(tr.groupby("trade_date").agg(trades=("price", "size"), first_ts=("ts", "min"), last_ts=("ts", "max")))
if not BUYSELL_IS_AGGRESSOR: warnings.warn("BUYSELL side не подтверждён как aggressor side: интерпретируй TFI только условно, не как поток агрессивных покупок/продаж.")


rows=3,097,105; duplicate IDs removed=0; days=33; unknown side=0.00%


,value
unique_trade_id,3097105
unique_ts,692950
duplicate_ts_rows,2783605
ts_monotonic,True
unique_prices,2423


,trades,first_ts,last_ts
trade_date,,,
2026-09-24,107817,2026-09-24 06:59:40+03:00,2026-09-24 20:35:12+03:00
2026-09-25,96821,2026-09-25 06:59:45+03:00,2026-09-25 21:42:33+03:00
2026-09-28,87391,2026-09-28 06:59:34+03:00,2026-09-28 20:51:48+03:00
2026-09-29,136651,2026-09-29 06:59:32+03:00,2026-09-29 22:31:34+03:00
2026-09-30,61039,2026-09-30 06:59:36+03:00,2026-09-30 21:16:45+03:00
2026-10-02,52711,2026-10-02 06:59:40+03:00,2026-10-02 20:47:05+03:00
2026-10-05,74127,2026-10-05 06:59:33+03:00,2026-10-05 22:30:26+03:00
2026-10-06,76171,2026-10-06 06:59:53+03:00,2026-10-06 21:47:19+03:00
2026-10-07,49263,2026-10-07 06:59:32+03:00,2026-10-07 23:33:12+03:00


C:\Users\Сергей Кравцов\AppData\Local\Temp\ipykernel_10984\1877361185.py:66: UserWarning: BUYSELL side не подтверждён как aggressor side: интерпретируй TFI только условно, не как поток агрессивных покупок/продаж.
  if not BUYSELL_IS_AGGRESSOR: warnings.warn("BUYSELL side не подтверждён как aggressor side: интерпретируй TFI только условно, не как поток агрессивных покупок/продаж.")


## 3. Trade Flow features
Rolling-окна включают текущую сделку i и предыдущие сделки, но **сбрасываются на границе торговой даты**. Если решение должно приниматься до исполнения i, сдвинь признаки на 1 строку.

In [6]:
def ratio(a, b):
    return (a / b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

feat = tr[["ts", "trade_date", "price", "size", "notional", "sign"]].copy()
feat["signed_size"] = tr.signed_size
feat["signed_notional"] = tr.signed_notional
feat["trade_return_bps"] = np.nan
feat["seconds_since_prev_trade"] = np.nan

# All rolling features reset at each trading date. This avoids mixing the
# last trades of yesterday with the first trades of today.
for day, idx in feat.groupby("trade_date", sort=False).groups.items():
    g = feat.loc[idx]
    price = g["price"]
    feat.loc[idx, "trade_return_bps"] = 1e4 * np.log(price / price.shift(1))
    feat.loc[idx, "seconds_since_prev_trade"] = g["ts"].diff().dt.total_seconds()

    for w in FEATURE_WINDOWS:
        r_size = g["size"].rolling(w, min_periods=w)
        r_signed_size = g["signed_size"].rolling(w, min_periods=w)
        r_notional = g["notional"].rolling(w, min_periods=w)
        r_signed_notional = g["signed_notional"].rolling(w, min_periods=w)
        r_sign = g["sign"].rolling(w, min_periods=w)

        # TFI volume = signed executed quantity / total executed quantity.
        # TFI notional = signed traded value / total traded value.
        # TFI count = signed trade count / number of trades with known side.
        feat.loc[idx, f"tfi_volume_{w}"] = ratio(
            r_signed_size.sum(), r_size.sum()
        ).to_numpy()
        feat.loc[idx, f"tfi_notional_{w}"] = ratio(
            r_signed_notional.sum(), r_notional.sum()
        ).to_numpy()
        feat.loc[idx, f"tfi_count_{w}"] = ratio(
            r_sign.sum(), r_sign.count()
        ).to_numpy()

        elapsed = (g["ts"] - g["ts"].shift(w - 1)).dt.total_seconds()
        feat.loc[idx, f"trade_intensity_{w}_per_sec"] = ratio(
            pd.Series(float(w - 1), index=g.index), elapsed
        ).to_numpy()
        feat.loc[idx, f"median_gap_{w}_sec"] = (
            g["ts"].diff().dt.total_seconds().rolling(w, min_periods=w).median().to_numpy()
        )

# Directional run length also resets at every trading date.
feat["directional_run"] = 0.0
for day, idx in feat.groupby("trade_date", sort=False).groups.items():
    s = feat.loc[idx, "sign"].to_numpy(dtype=float)
    run = np.zeros(len(s), dtype=float)
    for i in range(len(s)):
        if not np.isfinite(s[i]):
            run[i] = 0.0
        elif i and np.isfinite(s[i-1]) and s[i] == s[i-1]:
            run[i] = run[i-1] + s[i]
        else:
            run[i] = s[i]
    feat.loc[idx, "directional_run"] = run

for w in FEATURE_WINDOWS:
    feat[f"directional_run_{w}"] = np.clip(feat["directional_run"], -w, w)

# TFI must be a bounded ratio. Fail loudly if this invariant is violated.
for w in FEATURE_WINDOWS:
    for col in (f"tfi_volume_{w}", f"tfi_notional_{w}", f"tfi_count_{w}"):
        x = feat[col].dropna()
        if len(x) and (x.abs() > 1.0000001).any():
            raise ValueError(
                f"{col} вышел за [-1, 1]. Проверь sign, size/notional и исходные данные."
            )

feature_cols = [c for c in feat if c.startswith(("tfi_", "trade_intensity_", "median_gap_", "directional_run_"))]
display(feat[feature_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
tfi_volume_10,3096808.0,-0.007226,0.840149,-1.000000,-0.962264,-0.022222,0.957265,1.0
tfi_notional_10,3096808.0,-0.007209,0.840149,-1.000000,-0.962259,-0.022179,0.957269,1.0
tfi_count_10,3096808.0,0.017621,0.733163,-1.000000,-0.800000,0.000000,0.800000,1.0
trade_intensity_10_per_sec,2023714.0,3.309971,3.095569,0.038136,0.900000,1.800000,4.500000,9.0
median_gap_10_sec,3096775.0,0.210516,0.592891,0.000000,0.000000,0.000000,0.000000,18.0
tfi_volume_25,3096313.0,-0.011652,0.751075,-1.000000,-0.793103,-0.024134,0.770074,1.0
tfi_notional_25,3096313.0,-0.011623,0.751075,-1.000000,-0.793068,-0.024062,0.770097,1.0
tfi_count_25,3096313.0,0.017606,0.631370,-1.000000,-0.520000,0.040000,0.520000,1.0
trade_intensity_25_per_sec,2546769.0,5.703306,7.326481,0.059113,1.142857,2.400000,6.000000,24.0
median_gap_25_sec,3096280.0,0.122772,0.423243,0.000000,0.000000,0.000000,0.000000,13.0


## 4. Targets: future transaction-price response
Все будущие цели рассчитываются только внутри текущей торговой даты. Для каждого i рассматриваются сделки j>i. Различные изменения цены — переходы на новое значение transaction price; повторные сделки по той же цене пропускаются. Это всё ещё не mid-price и подвержено bid-ask bounce.

In [65]:
labels = pd.DataFrame(index=tr.index)
# Targets are calculated independently inside each trading date. No overnight
# or next-session price is allowed to enter a forward label.
ends = {}
for day, idx in tr.groupby("trade_date", sort=False).groups.items():
    ix = np.asarray(list(idx), dtype=int)
    p = tr.loc[ix, "price"].to_numpy(dtype=float)
    ts_ns = tr.loc[ix, "ts"].astype("int64").to_numpy()
    n = len(p)
    local_labels = pd.DataFrame(index=ix)

    for h in TRADE_HORIZONS:
        f = np.full(n, np.nan)
        if h < n:
            f[:n-h] = p[h:]
        local_labels[f"ret_next_{h}_trades_bps"] = 1e4 * (f / p - 1.0)

    next_change = np.full(n, -1, dtype=np.int64)
    next_different = n - 1
    for i in range(n-2, -1, -1):
        if p[i+1] != p[i]:
            next_different = i+1
        next_change[i] = next_different if p[next_different] != p[i] else -1

    def kth_distinct_after(i, k):
        j = next_change[i]
        step = 1
        while j >= 0 and step < k:
            j = next_change[j]
            step += 1
        return j if j >= 0 and step == k else -1

    for k in PRICE_CHANGE_HORIZONS:
        idx_local = np.array([kth_distinct_after(i, k) for i in range(n)], dtype=np.int64)
        y = np.full(n, np.nan)
        ok = idx_local >= 0
        y[ok] = 1e4 * (p[idx_local[ok]] / p[ok] - 1.0)
        local_labels[f"ret_after_{k}_distinct_price_changes_bps"] = y
        ends[k] = None  # retained only as a marker; global overlap rebuilt below
        if k == 1:
            local_labels["trades_until_next_price_change"] = np.where(
                ok, idx_local - np.arange(n), np.nan
            )
            local_labels["seconds_until_next_price_change"] = np.where(
                ok, (ts_ns[np.maximum(idx_local, 0)] - ts_ns) / 1e9, np.nan
            )

    local_labels["next_price_change_direction"] = np.sign(
        local_labels["ret_after_1_distinct_price_changes_bps"]
    )
    labels.loc[ix, local_labels.columns] = local_labels

display(labels.head(12))


,ret_next_1_trades_bps,ret_next_3_trades_bps,ret_next_5_trades_bps,ret_next_10_trades_bps,ret_next_25_trades_bps,ret_next_50_trades_bps,ret_after_1_distinct_price_changes_bps,trades_until_next_price_change,seconds_until_next_price_change,ret_after_2_distinct_price_changes_bps,ret_after_3_distinct_price_changes_bps,ret_after_5_distinct_price_changes_bps,next_price_change_direction
0,0.0,0.0,0.0,0.0,0.000000,23.885350,31.847134,29.0,6.0,39.808917,23.88535,23.88535,1.0
1,0.0,0.0,0.0,0.0,0.000000,23.885350,31.847134,28.0,6.0,39.808917,23.88535,23.88535,1.0
2,0.0,0.0,0.0,0.0,0.000000,31.847134,31.847134,27.0,6.0,39.808917,23.88535,23.88535,1.0
3,0.0,0.0,0.0,0.0,0.000000,23.885350,31.847134,26.0,6.0,39.808917,23.88535,23.88535,1.0
4,0.0,0.0,0.0,0.0,31.847134,23.885350,31.847134,25.0,6.0,39.808917,23.88535,23.88535,1.0
5,0.0,0.0,0.0,0.0,31.847134,31.847134,31.847134,24.0,6.0,39.808917,23.88535,23.88535,1.0
6,0.0,0.0,0.0,0.0,31.847134,23.885350,31.847134,23.0,6.0,39.808917,23.88535,23.88535,1.0
7,0.0,0.0,0.0,0.0,31.847134,23.885350,31.847134,22.0,6.0,39.808917,23.88535,23.88535,1.0
8,0.0,0.0,0.0,0.0,31.847134,23.885350,31.847134,21.0,6.0,39.808917,23.88535,23.88535,1.0
9,0.0,0.0,0.0,0.0,31.847134,15.923567,31.847134,20.0,6.0,39.808917,23.88535,23.88535,1.0


## 5. Overlap и AFML average uniqueness
Интервал события — от i до последней сделки, использованной в его target (включительно). Считаем concurrency и среднюю uniqueness. Эти веса сохраняем для дальнейшей модели, но пока не используем как веса описательной статистики.

In [66]:
def uniqueness_weights(starts, ends, n_obs):
    starts = np.asarray(starts, dtype=int); ends = np.asarray(ends, dtype=int)
    valid = (ends >= starts) & (ends < n_obs)
    a, b = starts[valid], ends[valid]
    if not len(a): return np.array([]), np.zeros(n_obs, dtype=int)
    d = np.zeros(n_obs+1, dtype=np.int32)
    np.add.at(d, a, 1); np.add.at(d, b+1, -1)
    conc = np.cumsum(d[:-1])
    inv = np.divide(1., conc, out=np.zeros(n_obs), where=conc>0)
    cs = np.r_[0., np.cumsum(inv)]
    w = (cs[b+1]-cs[a])/(b-a+1)
    return w, conc

overlap_rows=[]
uniqueness=pd.DataFrame(index=tr.index)
# Fixed-horizon intervals are clipped to the current session.
for h in TRADE_HORIZONS:
    all_starts=[]; all_ends=[]
    for day, idx in tr.groupby("trade_date", sort=False).groups.items():
        ix=np.asarray(list(idx),dtype=int); m=len(ix)
        local=np.arange(m)
        ok=local+h<m
        all_starts.extend(ix[ok]); all_ends.extend(ix[ok]+h)
    a=np.asarray(all_starts,dtype=int); b=np.asarray(all_ends,dtype=int)
    w,c=uniqueness_weights(a,b,len(tr))
    col=f"uniqueness_fixed_{h}"
    uniqueness[col]=np.nan
    if len(a): uniqueness.loc[a,col]=w
    overlap_rows.append({"target":f"next_{h}_trades","events":len(a),
                         "mean_uniqueness":np.mean(w) if len(w) else np.nan,
                         "median_uniqueness":np.median(w) if len(w) else np.nan,
                         "max_concurrency":int(c.max())})

# Distinct-price event spans use the already calculated within-day trade count.
for k in PRICE_CHANGE_HORIZONS:
    if k == 1:
        distance = labels["trades_until_next_price_change"].to_numpy(dtype=float)
    else:
        # Build the exact k-th distinct-change endpoint per session.
        distance = np.full(len(tr), np.nan)
        for day, idx in tr.groupby("trade_date", sort=False).groups.items():
            ix=np.asarray(list(idx),dtype=int)
            pp=tr.loc[ix,"price"].to_numpy(dtype=float); m=len(pp)
            nxt=np.full(m,-1,dtype=np.int64); last=m-1
            for j in range(m-2,-1,-1):
                if pp[j+1]!=pp[j]: last=j+1
                nxt[j]=last if pp[last]!=pp[j] else -1
            for j in range(m):
                q=nxt[j]; step=1
                while q>=0 and step<k:
                    q=nxt[q]; step+=1
                if q>=0 and step==k: distance[ix[j]]=q-j
    starts=np.flatnonzero(np.isfinite(distance))
    ends_idx=starts+distance[starts].astype(int)
    w,c=uniqueness_weights(starts,ends_idx,len(tr))
    col=f"uniqueness_distinct_{k}"
    uniqueness[col]=np.nan
    if len(starts): uniqueness.loc[starts,col]=w
    overlap_rows.append({"target":f"after_{k}_distinct_changes","events":len(starts),
                         "mean_uniqueness":np.mean(w) if len(w) else np.nan,
                         "median_uniqueness":np.median(w) if len(w) else np.nan,
                         "max_concurrency":int(c.max())})

overlap_report=pd.DataFrame(overlap_rows)
display(overlap_report)

,target,events,mean_uniqueness,median_uniqueness,max_concurrency
0,next_1_trades,1516074,0.500008,0.500000,2
1,next_3_trades,1516026,0.250012,0.250000,4
2,next_5_trades,1515978,0.166680,0.166667,6
3,next_10_trades,1515858,0.090923,0.090909,11
4,next_25_trades,1515498,0.038477,0.038462,26
5,next_50_trades,1514898,0.019623,0.019608,51
6,after_1_distinct_changes,1515993,0.186778,0.158631,1121
7,after_2_distinct_changes,1515876,0.100875,0.087940,1122
8,after_3_distinct_changes,1515772,0.068380,0.061352,1659
9,after_5_distinct_changes,1515556,0.041473,0.038555,1794


## 6. Bucket analysis
Сравниваем нижний/верхний квинтиль TFI. Это exploratory conditional statistics, не тест готовой стратегии.

In [67]:
analysis = pd.concat([feat, labels], axis=1)
target_cols = [c for c in ["ret_next_1_trades_bps", "ret_next_5_trades_bps", "ret_next_10_trades_bps", "ret_after_1_distinct_price_changes_bps", "ret_after_3_distinct_price_changes_bps", "ret_after_5_distinct_price_changes_bps"] if c in analysis]
def bucket_summary(df, signal, targets, q=5):
    x=df[signal]; ok=x.notna()
    if ok.sum()<100: return pd.DataFrame()
    b=pd.qcut(x[ok].rank(method="first"), q=q, labels=False, duplicates="drop")
    z=df.loc[ok, targets].copy(); z["bucket"]=b
    rows=[]
    for bucket,g in z.groupby("bucket"):
        row={"bucket":int(bucket),"n":len(g)}
        for col in targets:
            row[col+"_mean"]=g[col].mean(); row[col+"_median"]=g[col].median(); row[col+"_up_prob"]=(g[col]>0).mean()
        rows.append(row)
    return pd.DataFrame(rows)

parts=[]
for sig in [f"tfi_volume_{w}" for w in FEATURE_WINDOWS]+[f"tfi_notional_{w}" for w in FEATURE_WINDOWS]:
    tab=bucket_summary(analysis,sig,target_cols)
    if len(tab): tab.insert(0,"signal",sig); parts.append(tab)
bucket_results=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
display(bucket_results)


,signal,bucket,n,ret_next_1_trades_bps_mean,ret_next_1_trades_bps_median,ret_next_1_trades_bps_up_prob,ret_next_5_trades_bps_mean,ret_next_5_trades_bps_median,ret_next_5_trades_bps_up_prob,ret_next_10_trades_bps_mean,ret_next_10_trades_bps_median,ret_next_10_trades_bps_up_prob,ret_after_1_distinct_price_changes_bps_mean,ret_after_1_distinct_price_changes_bps_median,ret_after_1_distinct_price_changes_bps_up_prob,ret_after_3_distinct_price_changes_bps_mean,ret_after_3_distinct_price_changes_bps_median,ret_after_3_distinct_price_changes_bps_up_prob,ret_after_5_distinct_price_changes_bps_mean,ret_after_5_distinct_price_changes_bps_median,ret_after_5_distinct_price_changes_bps_up_prob
0,tfi_volume_10,0,303177,0.151267,0.0,0.071730,0.188049,0.0,0.180578,0.117881,0.0,0.239665,0.692819,-3.016591,0.459082,0.074995,0.0,0.470484,-0.049855,0.0,0.455988
1,tfi_volume_10,1,303176,-0.038852,0.0,0.122717,-0.182332,0.0,0.239109,-0.289400,0.0,0.283812,-0.336712,-3.012048,0.465766,-0.445963,0.0,0.442621,-0.528163,0.0,0.433992
2,tfi_volume_10,2,303176,0.009855,0.0,0.129822,0.022446,0.0,0.268399,0.035828,0.0,0.315668,-0.068817,-2.904444,0.498070,-0.007640,0.0,0.459845,0.021981,0.0,0.451032
3,tfi_volume_10,3,303176,0.062727,0.0,0.129967,0.181819,0.0,0.267264,0.310811,0.0,0.324762,0.072230,2.972652,0.520322,0.400222,0.0,0.481034,0.596756,0.0,0.475315
4,tfi_volume_10,4,303177,-0.173063,0.0,0.061984,-0.156874,0.0,0.211365,-0.071725,0.0,0.288884,-0.246919,3.014772,0.541050,0.116445,0.0,0.455648,0.465816,0.0,0.454177
5,tfi_volume_25,0,303105,0.016128,0.0,0.075812,-0.080173,0.0,0.170891,-0.199811,0.0,0.221036,0.370036,-3.039514,0.448541,-0.357052,0.0,0.446360,-0.488870,0.0,0.433622
6,tfi_volume_25,1,303104,-0.004629,0.0,0.116547,-0.073434,0.0,0.244698,-0.138070,0.0,0.292629,-0.197289,-2.977963,0.476094,-0.200155,0.0,0.451871,-0.278318,0.0,0.441113
7,tfi_volume_25,2,303104,0.000373,0.0,0.121889,-0.008298,0.0,0.262771,-0.006615,0.0,0.314120,-0.097800,-2.923977,0.495137,-0.053827,0.0,0.456002,0.018369,0.0,0.452874
8,tfi_volume_25,3,303104,0.015495,0.0,0.121886,0.104804,0.0,0.272240,0.176980,0.0,0.328805,0.022830,2.961208,0.516671,0.244490,0.0,0.471439,0.368949,0.0,0.463933
9,tfi_volume_25,4,303105,-0.015354,0.0,0.080101,0.110332,0.0,0.216163,0.270652,0.0,0.296217,0.016706,3.042288,0.547912,0.506201,0.0,0.483974,0.888602,0.0,0.478966


## 7. Day-block bootstrap
Bootstrap целыми торговыми днями сохраняет внутридневную зависимость. Для ускорения ресэмплируются предварительно агрегированные дневные суммы/счётчики, а не миллионы строк. При небольшом числе дней интервалы остаются грубыми.

In [68]:
def day_block_bootstrap(df, signal, target, reps=BOOTSTRAP_REPS, seed=63):
    d=df[["trade_date",signal,target]].dropna()
    days=d.trade_date.unique()
    if len(days)<3:
        return {"days":len(days),"spread_bps":np.nan,"ci_low":np.nan,"ci_high":np.nan}

    lo,hi=d[signal].quantile([.2,.8])
    d=d.copy()
    d["low_sum"]=np.where(d[signal]<=lo,d[target],0.0)
    d["low_n"]=(d[signal]<=lo).astype(np.int64)
    d["high_sum"]=np.where(d[signal]>=hi,d[target],0.0)
    d["high_n"]=(d[signal]>=hi).astype(np.int64)
    daily=d.groupby("trade_date",sort=False)[
        ["low_sum","low_n","high_sum","high_n"]
    ].sum()
    a=daily.to_numpy(dtype=float)

    total=a.sum(axis=0)
    point=total[2]/total[3]-total[0]/total[1]
    rng=np.random.default_rng(seed)
    # Resample compact daily aggregates, not millions of individual rows.
    draws=rng.integers(0,len(a),size=(reps,len(a)))
    sampled=a[draws].sum(axis=1)
    valid=(sampled[:,1]>0)&(sampled[:,3]>0)
    vals=sampled[valid,2]/sampled[valid,3]-sampled[valid,0]/sampled[valid,1]
    return {"days":len(days),"spread_bps":point,
            "ci_low":np.quantile(vals,.025) if len(vals) else np.nan,
            "ci_high":np.quantile(vals,.975) if len(vals) else np.nan,
            "valid_reps":len(vals)}

boot=[]
for sig in [f"tfi_volume_{w}" for w in FEATURE_WINDOWS]+[f"tfi_notional_{w}" for w in FEATURE_WINDOWS]:
    for target in target_cols:
        boot.append({"signal":sig,"target":target,**day_block_bootstrap(analysis,sig,target,BOOTSTRAP_REPS,RANDOM_SEED+len(boot))})
bootstrap_results=pd.DataFrame(boot); display(bootstrap_results)


,signal,target,days,spread_bps,ci_low,ci_high,valid_reps
0,tfi_volume_10,ret_next_1_trades_bps,24,-0.325256,-0.394416,-0.277131,500
1,tfi_volume_10,ret_next_5_trades_bps,24,-0.340605,-0.446874,-0.205240,500
2,tfi_volume_10,ret_next_10_trades_bps,24,-0.180648,-0.416287,0.157406,500
3,tfi_volume_10,ret_after_1_distinct_price_changes_bps,24,-0.954892,-1.851947,-0.183726,500
4,tfi_volume_10,ret_after_3_distinct_price_changes_bps,24,0.048265,-0.539600,0.748047,500
5,tfi_volume_10,ret_after_5_distinct_price_changes_bps,24,0.508702,-0.236312,1.348092,500
6,tfi_volume_25,ret_next_1_trades_bps,24,-0.031482,-0.061033,-0.011044,500
7,tfi_volume_25,ret_next_5_trades_bps,24,0.190510,0.096523,0.313984,500
8,tfi_volume_25,ret_next_10_trades_bps,24,0.470454,0.282149,0.729593,500
9,tfi_volume_25,ret_after_1_distinct_price_changes_bps,24,-0.353402,-1.119779,0.391961,500


## 8. Контроль прошлой доходности
TFI может отражать уже начавшееся движение. Разбиваем прошлую доходность на терцили и внутри них — TFI на терцили. Это диагностический контроль, не causal identification.

In [69]:
analysis["past_return_5_trades_bps"] = 1e4*np.log(tr.price/tr.price.shift(5))
analysis["past_return_20_trades_bps"] = 1e4*np.log(tr.price/tr.price.shift(20))
def momentum_table(df, signal, target, momentum, q=3):
    x=df[[signal,target,momentum]].dropna().copy()
    if len(x)<100: return pd.DataFrame()
    x["momentum_bucket"]=pd.qcut(x[momentum].rank(method="first"),q,labels=False,duplicates="drop")
    x["flow_bucket"]=pd.qcut(x[signal].rank(method="first"),q,labels=False,duplicates="drop")
    return x.groupby(["momentum_bucket","flow_bucket"],observed=True).agg(n=(target,"size"),mean_future_bps=(target,"mean"),median_future_bps=(target,"median"),up_probability=(target,lambda s:(s>0).mean())).reset_index()

mt=[]
for sig in [f"tfi_volume_{w}" for w in FEATURE_WINDOWS]:
    for target in ["ret_next_5_trades_bps","ret_after_1_distinct_price_changes_bps"]:
        for mom in ["past_return_5_trades_bps","past_return_20_trades_bps"]:
            tab=momentum_table(analysis,sig,target,mom)
            if len(tab): tab.insert(0,"momentum_control",mom); tab.insert(0,"target",target); tab.insert(0,"signal",sig); mt.append(tab)
conditional_results=pd.concat(mt,ignore_index=True) if mt else pd.DataFrame(); display(conditional_results)


,signal,target,momentum_control,momentum_bucket,flow_bucket,n,mean_future_bps,median_future_bps,up_probability
0,tfi_volume_10,ret_next_5_trades_bps,past_return_5_trades_bps,0,0,214758,0.780078,0.000000,0.241826
1,tfi_volume_10,ret_next_5_trades_bps,past_return_5_trades_bps,0,1,186360,1.214273,0.000000,0.339885
2,tfi_volume_10,ret_next_5_trades_bps,past_return_5_trades_bps,0,2,104136,1.088302,0.000000,0.302268
3,tfi_volume_10,ret_next_5_trades_bps,past_return_5_trades_bps,1,0,175551,-0.082825,0.000000,0.166271
4,tfi_volume_10,ret_next_5_trades_bps,past_return_5_trades_bps,1,1,127736,-0.107178,0.000000,0.226827
...,...,...,...,...,...,...,...,...,...
175,tfi_volume_1000,ret_after_1_distinct_price_changes_bps,past_return_20_trades_bps,1,1,158888,-0.031699,-2.937720,0.492674
176,tfi_volume_1000,ret_after_1_distinct_price_changes_bps,past_return_20_trades_bps,1,2,164733,0.236975,2.931692,0.513188
177,tfi_volume_1000,ret_after_1_distinct_price_changes_bps,past_return_20_trades_bps,2,0,145402,-1.372642,-3.124024,0.427862
178,tfi_volume_1000,ret_after_1_distinct_price_changes_bps,past_return_20_trades_bps,2,1,169991,-1.117722,-3.073141,0.462189


## 9. Экспорт


In [70]:
tr.to_parquet(OUTPUT_DIR/f"{TICKER}_trades_normalized.parquet",index=False)
feat.to_parquet(OUTPUT_DIR/f"{TICKER}_trade_flow_features.parquet",index=False)
labels.to_parquet(OUTPUT_DIR/f"{TICKER}_trade_price_targets.parquet",index=False)
uniqueness.to_parquet(OUTPUT_DIR/f"{TICKER}_event_uniqueness.parquet",index=False)
overlap_report.to_csv(OUTPUT_DIR/f"{TICKER}_overlap_report.csv",index=False,encoding="utf-8-sig")
bucket_results.to_csv(OUTPUT_DIR/f"{TICKER}_bucket_results.csv",index=False,encoding="utf-8-sig")
bootstrap_results.to_csv(OUTPUT_DIR/f"{TICKER}_day_block_bootstrap.csv",index=False,encoding="utf-8-sig")
conditional_results.to_csv(OUTPUT_DIR/f"{TICKER}_momentum_control.csv",index=False,encoding="utf-8-sig")
print("Saved:",OUTPUT_DIR.resolve())


Saved: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\trade_flow_outputs_SMLT


## Как интерпретировать первый результат
- Если эффект есть только на `next_1_trade`, но исчезает на `next_1_distinct_price_changes`, вероятно, наблюдаем bid-ask bounce.
- Если эффект остаётся на 1/3/5 distinct price changes, это интереснее, но всё ещё не mid-price alpha.
- Сравни окна 25/50/100 и обязательно контролируй прошлую доходность.
- Проверь, что выгрузка не обрезана API лимитом (в предыдущей диагностике было 500 000 строк — это мог быть лимит выборки, а не полная история).
- Для настоящего mid-price исследования нужна синхронная история best bid/ask updates. Редкие orderbook snapshots этого не заменяют.
